<a href="https://colab.research.google.com/github/KIMCHEN99/JAVA-Class/blob/main/5_2_boston_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 라이브러리 환경

In [1]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf
print(tf.__version__)

2.19.0


In [2]:
# 랜덤 시드 고정
SEED=12
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("시드 고정: ", SEED)

시드 고정:  12


In [3]:
housing = pd.read_csv('/content/BostonHousing.csv')
housing.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/boston.csv'

In [ ]:
# 2. 데이터 확인
print("shape:", housing.shape)        # (506, 14)
print(housing.head())

# 3. 피처(X)와 타겟(y) 분리
x_data = housing.drop(columns=["medv"])   # 13개 피처
y_data = housing["medv"]                   # 목적변수 MEDV

# 4. 확인
print("\nx_data shape:", x_data.shape)   # (506, 13)
print("y_data shape:", y_data.shape)     # (506,)

print("\nx_data 컬럼:", x_data.columns.tolist())
print("y_data 미리보기:\n", y_data.head())

In [ ]:
##피쳐 스케일링
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_data_scaled = scaler.fit_transform(x_data)
X_data_scaled[0]

# 데이터 전처리

In [ ]:
# 학습 - 테스트 데이터셋 분할
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_data_scaled, y_data,
                                                    test_size=0.2,
                                                    shuffle=True,
                                                    random_state=SEED)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# 신경망 학습

In [ ]:
# 심층 신경망
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
##심층 신경망 모델(MLP)
model = Sequential([
    Dense(128, activation='relu', input_dim=13),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])



In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])




In [ ]:

model.summary()

In [ ]:
# 모델 훈련
model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=2)

In [ ]:
# 평가
model.evaluate(X_test, y_test)

# 교차 검증

In [ ]:
history = model.fit(X_train, y_train, batch_size=32, epochs=200,
                    validation_split=0.25, verbose=2)

In [ ]:
import matplotlib.pyplot as plt
def plot_loss_curve(total_epoch=10, start=1):
    plt.figure(figsize=(5, 5))
    plt.plot(range(start, total_epoch + 1),
             history.history['loss'][start-1:total_epoch],
             label='Train')
    plt.plot(range(start, total_epoch + 1),
             history.history['val_loss'][start-1:total_epoch],
             label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('mse')
    plt.legend()
    plt.show()

plot_loss_curve(total_epoch=200, start=1)

In [ ]:
plot_loss_curve(total_epoch=200, start=20)

##DroupOut(0.3) 30%뉴런을 랜덤으로  off
##EarlySttopping 불필요한 학습 중단
##

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 1. 데이터 분리 먼저
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data,
                                                     test_size=0.2,
                                                     shuffle=True,
                                                     random_state=SEED)

# 2. 스케일링 (train fit, test transform)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. 모델 구성 (Dropout 추가)
model = Sequential([
    Dense(128, activation='relu', input_dim=13),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

# 4. 컴파일
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mse'])

# 5. EarlyStopping 적용
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# 6. 학습
history = model.fit(X_train_scaled, y_train,
                    batch_size=32,
                    epochs=200,
                    validation_split=0.25,
                    callbacks=[early_stop],
                    verbose=2)

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curve(total_epoch=None, start=1):

    # 실제 학습된 에폭 수 자동 감지 (EarlyStopping 대응)
    actual_epochs = len(history.history['loss'])
    if total_epoch is None:
        total_epoch = actual_epochs

    plt.figure(figsize=(5, 5))
    plt.plot(range(start, total_epoch + 1),
             history.history['loss'][start-1:total_epoch],
             label='Train', color='blue')
    plt.plot(range(start, total_epoch + 1),
             history.history['val_loss'][start-1:total_epoch],
             label='Validation', color='orange')

    plt.title('Train vs Validation Loss (MSE)')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# 호출
plot_loss_curve()

In [ ]:
plot_loss_curve(total_epoch=None, start=10)

In [ ]:
import matplotlib.pyplot as plt
def plot_loss_curve(total_epoch=10, start=1):
    plt.figure(figsize=(5, 5))
    plt.plot(range(start, total_epoch + 1),
             history.history['loss'][start-1:total_epoch],
             label='Train')
    plt.plot(range(start, total_epoch + 1),
             history.history['val_loss'][start-1:total_epoch],
             label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('mse')
    plt.legend()
    plt.show()

plot_loss_curve(total_epoch=len(history.history['loss']), start=1)

In [ ]:
plot_loss_curve(total_epoch=len(history.history['loss']), start=20)